# Strand C (instrument): Wayback static parse vs HTTP Archive, both June 2022

**Purpose:** the coverage strand uses Wayback snapshots to measure domains
nothing else measured. Before trusting those measurements, this notebook
quantifies what a *static* parse of archived HTML sees relative to a full
request log — on the same domains, in the same month (June 2022). Wayback
static parsing misses trackers injected by JavaScript at runtime, so it should
undercount; the question is by how much, and whether *presence* is still
informative enough to support a calibrated fill.

Also reports snapshot fidelity: how many targets had a usable capture, and how
far capture dates sit from the browsing window's midpoint.

Same classifier on both sides (pinned DuckDuckGo Tracker Radar), so rows
compare visibility, not list versions.

In [1]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath(".."))

import config
from utilities import pandas_to_tex, save_mpl_fig

PAIRS = [
    ("ddg_join_ads", "Ad trackers (static subset vs full request map)"),
    ("ddg_known_trackers", "Any known tracker"),
    ("fb_pixel", "Facebook requests"),
    ("google_analytics", "Google Analytics/GTM"),
    ("n_third_parties", "Any third party"),
]

Checking that all paths exist:
{'web_mobile': False, 'web_desktop': False, 'web': False, 'yg_profile': False, 'blacklight': False, 'who': False}


In [2]:
manifest = pd.read_csv(config.FP_WB_MANIFEST)
wb = pd.read_csv(config.FP_WB_DOMAIN_MEASURES)
ha22 = (
    pd.read_csv(config.FP_HA_DOMAIN_MEASURES)
    .query("crawl == 'panel'")
    .groupby("private_domain")[[k for k, _ in PAIRS]]
    .mean()
)
manifest.groupby("group")["status"].value_counts(dropna=False)

group      status     
calib_bl   ok              386
           no_capture       83
           cdx_error        28
           fetch_error       3
calib_ha   ok              410
           no_capture       70
           cdx_error        19
           fetch_error       1
remainder  no_capture     1660
           ok              584
           cdx_error        10
Name: count, dtype: int64

## Snapshot fidelity

In [3]:
ok = manifest.query("status == 'ok'")
by_group = manifest.groupby("group").agg(
    n_targets=("private_domain", "nunique"),
    pct_with_snapshot=("status", lambda s: 100 * (s == "ok").mean()),
).round(1)
print(by_group)

ts = pd.to_datetime(ok["snapshot_ts"].astype("Int64").astype(str), format="%Y%m%d%H%M%S")
days_off = (ts - pd.Timestamp("2022-06-15")).dt.days
print("\nsnapshot distance from 2022-06-15 (days):")
print(days_off.describe().round(1).to_string())

           n_targets  pct_with_snapshot
group                                  
calib_bl         500               77.2
calib_ha         500               82.0
remainder       2254               25.9

snapshot distance from 2022-06-15 (days):
count    1380.0
mean        1.4
std        19.0
min       -75.0
25%        -3.0
50%         0.0
75%        10.0
max        74.0


## Agreement with HTTP Archive, June 2022 (calib_ha group)

In [4]:
calib_ha = set(manifest.query("group == 'calib_ha'")["private_domain"])
j = (
    wb[wb["private_domain"].isin(calib_ha)]
    .merge(ha22, left_on="private_domain", right_index=True,
           suffixes=("_wb", "_ha"))
)
print(f"domains measured by both, June 2022: {len(j):,}")

rows = []
for k, label in PAIRS:
    pw, ph = (j[f"{k}_wb"] > 0), (j[f"{k}_ha"] > 0)
    rows.append({
        "measure": label,
        "n_domains": len(j),
        "wb_prev_pct": 100 * pw.mean(),
        "ha_prev_pct": 100 * ph.mean(),
        "agree_pct": 100 * (pw == ph).mean(),
        "recall_of_ha_pct": 100 * (pw & ph).sum() / max(ph.sum(), 1),
        "spearman_counts": j[f"{k}_wb"].corr(j[f"{k}_ha"], method="spearman"),
    })
agree = pd.DataFrame(rows)
agree.round(2)

domains measured by both, June 2022: 410


,measure,n_domains,wb_prev_pct,ha_prev_pct,agree_pct,recall_of_ha_pct,spearman_counts
0,Ad trackers (static subset vs full request map),410,89.27,96.59,92.20,92.17,0.37
1,Any known tracker,410,94.88,100.00,94.88,94.88,0.45
2,Facebook requests,410,36.59,56.34,59.76,46.75,0.24
3,Google Analytics/GTM,410,70.00,91.22,76.83,75.67,0.40
4,Any third party,410,96.59,100.00,96.59,96.59,0.44


In [5]:
tex = agree.copy()
tex["n_domains"] = tex["n_domains"].map("{:,}".format)
for c in ["wb_prev_pct", "ha_prev_pct", "agree_pct", "recall_of_ha_pct"]:
    tex[c] = tex[c].map("{:.1f}".format)
tex["spearman_counts"] = tex["spearman_counts"].map("{:.2f}".format)
pandas_to_tex(tex, os.path.join(config.TABLES_DIR, "wb_ha_agreement"))
print("wrote tables/wb_ha_agreement.tex")

wrote tables/wb_ha_agreement.tex


## Reading the results

`recall_of_ha_pct` is the key column: of domains where the full June-2022
request log shows the measure, what share does the static parse also flag?
Low recall with decent precision still supports a *presence-calibrated* fill
(what `06_coverage_bounds_wayback` uses) — the calibration conditions on WB
status, so systematic undercounting is absorbed into the conditional means.
If both recall and the count correlation are near zero for a measure, the
Wayback fill adds no information for it and the bounds notebook's `wb`
scenarios should be read as no better than assumption for that row.